## Text Summarizer Plugin

### Introduction
This notebook demonstrates how a Chrome plugin, built using Flask, leverages an OpenVINO™ backend to efficiently summarize any webpage via a URL or any PDF via an upload. The plugin utilizes Langchain tools for tasks such as text splitting and managing a vectorstore.

### Install Prerequisites
All the necessary prerequisites needed is mentioned in the Readme section

### Code Walkthrough
In this chrome plugin we have divided into two parts 
* **Backend** - In the backend, we have two python files code.py and server.py
* **Extension** - In the extension we have the front end code for the plugin (popup.html,popup.js,style.css)

In this notebook, we have merged the two python files(code.py and server.py) into one notebook for a clear understanding and will understand only how a backend works, front end part is explained in the extension folder. In the beginning we will walk through the main code(code.py) and then the server side code(server.py).

### 1. Main code for Summarization 

### Importing the necessary libraries

In [ ]:
from transformers import AutoTokenizer, pipeline
from optimum.intel import OVModelForCausalLM
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader

### Prompt Templates for Summarization & Question Answering Bot
Here we have created two variables for prompt template so that it can be called later on , one template for summarization and one for query asked in the bot

In [ ]:
#prompt template for summarization
summary_template= """Write a concise summary of the following: "{context}" CONCISE SUMMARY: """
#prompt template for query
query_template="""Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Use 10 words maximum and keep the answer as concise as possible in one sentence.
    Always say "thanks for asking!" at the end of the answer.
 
    {context}
 
    Question: {question}
 
    Helpful Answer:"""

### Preprocessing 

* Loads page content from the webpage/PDF. Document loaders in RAG are used to load and preprocess the documents that will be used for retrieval     during the question answering process.
* Splits the page data using Recursive Character Text Splitter & creates embeddings using HuggingFace Embeddings. RecursiveCharacterTextSplitter is used to split text into smaller pieces recursively at the character level.
* In RAG, embeddings plays a crucial role in retrieval of relevant documents for a given query and Sentence Transformers helps to generate embeddings for each document in your knowledge base.
* This is further stored into ChromaDB for futher retrieval usage .Chroma is a vector store and embeddings database designed from the ground-up to make it easy to build AI applications with embeddings.

In [ ]:
#Function created for preprocessing 
def pre_processing(loader):    
    try:
        page_data = loader.load()
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
        all_splits = text_splitter.split_documents(page_data)
        embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        global vectorstore
        vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)  
        return vectorstore
    except Exception as e:
        print(f"Error while processing Webpage/PDF page content: {e}")

### Loading LLM models
In this we are trying to create a common function for loading the LLM models in a drop-down and then trying to return that LLM model which will be used later on for summarization

In [ ]:
#function created for laoding the LLM
def load_llm(model_id):
    if model_id:
        try:
            if model_id=="Meta LLama 2":
                model_path=r"<path of the meta llama 2 model>"
            elif model_id=="Qwen 7B Instruct":
                model_path=r"<path to Qwen 7B Instruct model"
            model = OVModelForCausalLM.from_pretrained(model_path , device='GPU')
            tokenizer = AutoTokenizer.from_pretrained(model_path)
            pipe=pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=4000,  
                device=model.device
            )
            global llm_model 
            llm_model = HuggingFacePipeline(pipeline=pipe)
            return llm_model
        except Exception as e:
            print(f"Failed to load the model. Please check whether the model_path is correct. \n Error: {e}")

###  URL Summarization
Here we try to load the web page when a user enters an URL into the plugin which in return loads the page data and passes into the RetrievalQA chain. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are using WebBaseLoader to load the documents from the web.
* The **WebBaseLoader** in Retrieval Augmented Generation (RAG) is a type of document loader that is designed to load documents from the web.The WebBaseLoader is used when the documents for retrieval are not stored locally or in a Hugging Face dataset, but are instead located on the web.
* **RetrievalQA** is a type of question answering system that uses a retriever to fetch relevant documents given a question, and then uses a reader to extract the answer from the retrieved documents.

In [ ]:
#function created for URL content summarization
def web_out(urls):
    try:
        loader = WebBaseLoader(urls)
        global summ_vectorstore 
        summ_vectorstore = pre_processing(loader)
        prompt = PromptTemplate(
            template=summary_template,
            input_variables=["context", "question"]
        )
    
        qa_chain = RetrievalQA.from_chain_type(
            llm=llm_model,
            retriever=summ_vectorstore.as_retriever(),
            chain_type="stuff",
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=False,
        )
        
        question = "Please summarize the context in one paragraph of 100 words"
        summary = qa_chain({'query': question})
        response = summary['result']
        summary_start = response.find("CONCISE SUMMARY:")
        concise_summary = response[summary_start + len("CONCISE SUMMARY:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Failed to summarize webpage \n Error: {e}")

### URL Question Answering BOT
The function defined below does a follow up questions to the bot related to the content being uploaded as URL post summarization. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it .
Here we are taking the **query template** which is declared as global and created a chain with llm model, retreiver and chain type as "stuff" and then based on the query asked by the user we get an answer from the LLM model.

In [ ]:
#function created for QnA bot for URL
def url_query(query):
    try:
        prompt = PromptTemplate(
            template=query_template,
            input_variables=["context", "question"]
            )
        reduce_chain = RetrievalQA.from_chain_type(
                llm=llm_model,
                retriever=summ_vectorstore.as_retriever(),
                chain_type="stuff",
                chain_type_kwargs={"prompt": prompt},
                return_source_documents=False
            )
        summary = reduce_chain({'query': query})
        summ_vectorstore.delete
        response = summary['result']
        summary_start = response.find("Helpful Answer:")
        concise_summary = response[summary_start + len("Helpful Answer:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Error in Webpage Summarizer QA BoT: {e}")


### PDF Summarization
Here we try to take a PDF file as input into the plugin which loads the page data and passes into the RetrievalQA chain. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are using PyPDF loader to load the PDF document.
* **PyPDFLoader** is a document loader within the LangChain framework specifically designed to handle PDF files. It allows you to extract text from PDF documents and load them into a format suitable for language models and other text-based applications.


In [ ]:
#function created for PDF summarization
def pdf_out(pdf):
    try:
        loader = PyPDFLoader(pdf, extract_images=False)
        global pdf_vectorstore
        pdf_vectorstore=pre_processing(loader)
    
        prompt = PromptTemplate(
            template=summary_template,
            input_variables=["context", "question"]
        )
        reduce_chain = RetrievalQA.from_chain_type(
            llm=llm_model,
            retriever=pdf_vectorstore.as_retriever(),
            chain_type="stuff",
            chain_type_kwargs={"prompt": prompt},
            return_source_documents=False,
        )
        question = "Please summarize the context in one paragraph of 60 words"
        summary = reduce_chain({'query': question})

        response = summary['result']
        summary_start = response.find("CONCISE SUMMARY:")
        concise_summary = response[summary_start + len("CONCISE SUMMARY:"):].strip()
        return concise_summary
    except Exception as e:
        print(f"Failed to summarize PDF \n Error: {e}")

### PDF Question Answering BOT
The function defined below does a follow up questions to the bot related to the content being uploaded as PDF post summarization. When a question is being asked in the retreival QA chain , we try to get a concise summary and return it . Here we are taking the **query template** which is declared as global and created a chain with llm model, retreiver and chain type as "stuff" and then based on the query asked by the user we get an answer from the LLM model.

In [ ]:
#function created for Question Answering bot for PDF
def pdf_query(query):
    try:
        prompt = PromptTemplate(
            template=query_template,
            input_variables=["context", "question"]
            )
        reduce_chain = RetrievalQA.from_chain_type(
                llm=llm_model,
                retriever=pdf_vectorstore.as_retriever(),
                chain_type="stuff",
                chain_type_kwargs={"prompt": prompt},
                return_source_documents=False
            )
        summary = reduce_chain({'query': query})
        response = summary['result']
        summary_start = response.find("Helpful Answer:")
        concise_summary = response[summary_start + len("Helpful Answer:"):].strip()
        print(concise_summary)

### 2. Server Code for TEXT SUMMARIZATION plugin

### Importing the necessary libraries

In [ ]:
#importing libraries
import time
from flask import Flask, Response, request, jsonify
from flask_cors import CORS
from code import load_llm, web_out, pdf_out,pdf_query,url_query
import tempfile
import chromadb

### Flask App Initialization and Enabling CORS
Here we are initializing a flask and enabling CORS which allows the flask app tobe accessed and interacted with from other domains and we are restricting the types of files that can be uploaded to the application.

In [ ]:
#Initializing the flask app and enabling CORS
app = Flask(__name__)
CORS(app)  # This will enable CORS for all routes
ALLOWED_EXTENSIONS = {'txt', 'pdf', 'png', 'jpg', 'jpeg', 'gif'}

### Loading the LLM model
Here we are loading the model once and this function will trigger the model compilation function present in the main code for summarization.

In [ ]:
#Loading model once
@app.route('/select-model', methods=['POST'])
def select_model():
    try:
        global current_model
        data = request.get_json()
        model_id = data.get('model_id')
        current_model = load_llm(model_id)
        return jsonify({'message': f'Model {model_id} loaded successfully.'}), 200
    
    except Exception as e:
        return jsonify({'message': f'Failed to load model \n Error: {e}'}), 500
        

### Generator Function 
This is a generator function which helps to Stream and yield the response content chunk by chunk

In [ ]:
#function explaining how a sentences are divided into chunks
def stream_output(process_function, *args):
    try:
        for chunk in process_function(*args):
            if chunk is not None:
                yield f"{chunk}"
    except Exception as e:
        yield f"Error while streaming output: {e}"

### URL Processing Code
This will fetch the URL from the user's input from the plugin and trigger the URL summarization function present in the main code 

In [ ]:
# URL processing code
@app.route('/process-url', methods=['POST'])
def process_url():
    try:
        data = request.get_json()
        url = data.get('url')  
        if not url:
            return jsonify({'message': 'No URL provided'}), 400
        chromadb.api.client.SharedSystemClient.clear_system_cache()
        return Response(stream_output(web_out, [url]), content_type='text/event-stream')
    
    except Exception as e: 
        print(f"Error while processing URL: {e}")
        return jsonify({'message': f'Error while processomg URL- {e}'}), 400

### PDF Processing Code
This function takes the PDF uploaded by the user and trigger the PDF summarization function present in the main code

In [ ]:
# PDF processing code
@app.route('/upload-pdf', methods=['POST'])
def upload_pdf():
    pdf_file = request.files['pdf'] 
    if pdf_file and pdf_file.content_type == 'application/pdf':
        try:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as temp_pdf:
                pdf_file.save(temp_pdf.name)
                temp_pdf_path = temp_pdf.name
                print(temp_pdf_path)
           
            chromadb.api.client.SharedSystemClient.clear_system_cache()
            return Response(stream_output(pdf_out, temp_pdf_path), content_type='text/event-stream')
 
        except Exception as e:
            return jsonify({"message": f"Error processing PDF: {str(e)}"}), 500
    else:
        return jsonify({"message": "Invalid file type. Please upload a PDF."}), 400
 

### PDF query code for Question Answering bot
Once the PDF content summarization is done user asks query to the Question Asnwering bot which gets triggered to the Query for the PDF function present in the main code

In [ ]:
#query code for pdf 
@app.route('/your_query_pdf', methods=['POST'])
def pdf_process_query():
    try:
        data = request.get_json()
        query=data.get('query')
        if not data:
            return jsonify({'message':'no query provided'}),400
        response_message=str(pdf_query(query))
        return jsonify({'message': response_message}), 200
    except Exception as e:
        return jsonify({'message': f'Error: {e}'}), 500

### URL query code for Question Answering bot
Once the URL content summarization is done user asks query to the Question Answering bot which gets triggered to the Query for the URL function present in the main code

In [ ]:
#query code for url
@app.route('/your_query_url', methods=['POST'])
def url_process_query():
    try:
        data = request.get_json()
        query=data.get('query')
        if not data:
            return jsonify({'message':'no query provided'}),400
        response_message=str(url_query(query))
        return jsonify({'message': response_message}), 200
    except Exception as e:
        return jsonify({'message': f'Error: {e}'}), 500



### Calling the main function
This code snippet ensures that the Flask development server starts only when the application is run directly

In [ ]:
#main function
if __name__ == '__main__':
    app.run(port=5000)